# Suitability workflow: from the skill's recipes to a map

This notebook runs the whole suitability workflow for one NbS family, with every step visible.

```
recipes (skill)  config/recipes/<nbs>/<family>/*.json   function + weight per variable
roles            config/roles.toml                      limiting or modulating
run              config/runs/*.toml + the cell below    area, grid, eligibility, sensitivity
        │
        ▼
1 recipes ─► 2 plan ─► 3 grid ─► 4 inputs ─► 5 scores z = f(x) ─► 6 L, G, M, S ─► 7 one pixel
          ─► 8 classes ─► 9 sensitivity ─► 10 outputs
```

**The index (approach 3 of the methodology)**

$$S = L \cdot G \cdot M$$

- **L**: the lowest score among the *limiting* variables (no weights). The most limiting factor rules, so good terrain never compensates for a climate where the practice does not work.
- **G**: 1 when the land cover is one where the NbS can be implemented, 0 when not.
- **M**: the weighted sum of the *modulating* scores. The recipe weights are renormalised among the modulating variables so they sum to 1.

A value outside the nodes of its recipe has no evidence behind it: the pixel is reported as **not evaluated**, never as 0. Edit only the parameters cell, then run all.

In [ ]:
# ===== PARAMETERS: edit only this cell =====
RUN_CONFIG = "config/runs/agroforestry_cross_family.toml"   # defaults; the values below override it

NBS = "agroforestry"                 # folder under config/recipes
FAMILY = "cross_family"              # folder under config/recipes/<NBS>

AOI = {"kind": "bbox", "name": "Haiti (approximate box)", "bbox": [-74.5, 18.0, -71.6, 20.1]}
# AOI = {"kind": "admin", "name": "Haiti", "admin_code": "HTI", "admin_level": 0}   # official boundary
RESOLUTION_M = 1000                  # grid cell size in metres

WEIGHTS = "recipe"                   # "recipe" | "equal"
OUTSIDE = "unknown"                  # beyond a recipe's nodes: "unknown" (not evaluated) | "clamp"
ELIGIBILITY = "apply"                # "apply": ESA WorldCover | "skip": every cell eligible (quick look)
ELIGIBLE_CLASSES = [30, 40]          # WorldCover classes where the NbS can be implemented (grassland, cropland)
MIN_SHARE = 0.5                      # share of a cell that must be in those classes

N_DRAWS = 300                        # sensitivity: Dirichlet draws of the modulating weights
CONCENTRATION = 50.0                 # larger keeps the draws closer to the recipe weights
SEED = 20261124

DATA_SOURCE = "real"                 # "real", or "synthetic" to test the notebook offline (never real results)
DATA_ORDER = ("local", "hf")         # where a dataset file is looked for, in this order
LOCAL_DATA_DIR = ""                  # optional folder with a copy of the dataset (raw/<folder>/<file>)
HF_MODE = "stream"                   # "stream": read only the window | "download": whole files once
EE_PROJECT = ""                      # empty: read EE_PROJECT from .env
USE_CACHE = True                     # Earth Engine layers are extracted once and kept in cache/layers/
CACHE_DIR = "cache/layers"
GEE_WORKERS = 4                      # Earth Engine requests in parallel
GEE_TILE_INPUT_PIXELS = 2_500_000    # lower it if you see "User memory limit exceeded"

PIXEL = None                         # (row, col) to explain in step 7; None picks the highest-scoring pixel
SAVE = True                          # write figures, rasters and the manifest to runs/<...>/

In [ ]:
import importlib.util
import sys
from pathlib import Path


def find_repo_root():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "src" / "nbs_runtime").is_dir() and (p / "pyproject.toml").exists():
            return p
    raise RuntimeError(f"Could not find the runtime repository above {here}. Open this notebook from inside it.")


REPO_ROOT = find_repo_root()
if importlib.util.find_spec("nbs_runtime") is None:
    sys.path.insert(0, str(REPO_ROOT / "src"))

needed = {"numpy": "numpy", "matplotlib": "matplotlib"}
if DATA_SOURCE == "real":
    needed |= {"rasterio": "rasterio", "ee": "earthengine-api", "xee": "xee", "xarray": "xarray"}
    if AOI.get("kind") == "admin":
        needed |= {"geopandas": "geopandas", "shapely": "shapely"}
missing = sorted({pip for mod, pip in needed.items() if importlib.util.find_spec(mod) is None})
if missing:
    raise ModuleNotFoundError(f"Missing in this Python: {', '.join(missing)}\n"
                              f"Install with:  uv pip install --python {sys.executable} {' '.join(missing)}\n"
                              "then restart the kernel.")
print("python:", sys.executable)
print("repository:", REPO_ROOT)

## 1. Recipes

The skill delivers one file per variable with the function (nodes `x` → `score`, linear between nodes) and a weight. The roles come from `config/roles.toml`. Limiting variables ignore their weight; the modulating weights are renormalised so they sum to 1.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from nbs_runtime import viz
from nbs_runtime.runconfig import RunConfig

cfg = RunConfig.load(REPO_ROOT / RUN_CONFIG, {
    "run": {"nbs": NBS, "family": FAMILY, "weights": WEIGHTS, "outside": OUTSIDE},
    "aoi": AOI,
    "grid": {"resolution_m": RESOLUTION_M},
    "eligibility": {"mode": ELIGIBILITY, "classes": ELIGIBLE_CLASSES, "min_share": MIN_SHARE},
    "sensitivity": {"n_draws": N_DRAWS, "concentration": CONCENTRATION, "seed": SEED},
    "data": {"order": list(DATA_ORDER), "hf_mode": HF_MODE},
})

from nbs_runtime.recipes import load_family

fam = load_family(cfg.run.nbs, cfg.run.family)
w = fam.modulating_weights(cfg.run.weights)
print(f"{fam.nbs}/{fam.family}: {len(fam.recipes)} variables\n")
print(f"{'variable':<24}{'role':<12}{'unit':<11}{'nodes':<22}{'weight (recipe)':>16}{'weight used':>13}  papers")
for r in fam.recipes:
    used = f"{w[r.variable]:.3f}" if r.variable in w else "-"
    rec = "-" if r.weight is None else f"{r.weight:g}"
    print(f"{r.variable:<24}{fam.roles[r.variable]:<12}{r.unit:<11}{f'{r.x[0]:g} to {r.x[-1]:g} ({len(r.x)})':<22}"
          f"{rec:>16}{used:>13}  {', '.join(r.papers)}")

In [ ]:
viz.plot_curves(fam)
plt.show()

## 2. Plan

What blocks the run (holds) and what to keep in mind when reading the results (warnings), checked before any data is read: every recipe loads, every variable has a role, a data source and the same unit as its recipe.

In [ ]:
from nbs_runtime.plan import plan

p = plan(cfg)
print("HOLDS (block the run):" if p.holds else "No holds.")
for h in p.holds:
    print("  -", h)
print("Warnings:" if p.warnings else "No warnings.")
for m in p.warnings:
    print("  -", m)
if p.holds:
    raise SystemExit("Resolve the holds above before running.")

## 3. Grid and data sources

In [ ]:
from nbs_runtime.data import gee
from nbs_runtime.data.cache import LayerCache
from nbs_runtime.data.variables_loader import LoadContext
from nbs_runtime.grid import Grid
from nbs_runtime.variables import BY_ID, GEE

boundary = cfg.aoi.boundary() if cfg.aoi.kind == "admin" else None
grid = Grid.from_aoi(cfg.aoi, cfg.grid.resolution_m)
synthetic = DATA_SOURCE == "synthetic"
OUT_DIR = REPO_ROOT / "runs" / f"{fam.nbs}-{fam.family}-{cfg.fingerprint()[:10]}-{DATA_SOURCE}"
if SAVE:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
fig_path = lambda name: (OUT_DIR / f"{name}.png") if SAVE else None
print(f"grid: {grid.shape[0]} x {grid.shape[1]} cells at {grid.resolution_m} m ({grid.shape[0] * grid.shape[1]:,} cells)")
print("outputs:", OUT_DIR.relative_to(REPO_ROOT) if SAVE else "not saved")

specs = [BY_ID[v] for v in fam.variables] + ([BY_ID["land_cover"]] if cfg.eligibility.mode == "apply" else [])
cache = LayerCache(REPO_ROOT / CACHE_DIR, grid, gee.ASSET_IDS) if (USE_CACHE and not synthetic) else None
local_roots = [REPO_ROOT / "data"] + ([Path(LOCAL_DATA_DIR)] if LOCAL_DATA_DIR else [])
ctx = LoadContext(grid, cfg.data.order, local_roots, cfg.data.hf_mode, cache, GEE_WORKERS, GEE_TILE_INPUT_PIXELS,
                  verbose=True, synthetic=synthetic, seed=SEED)
if not synthetic and any(s.source == GEE for s in specs):
    print("Earth Engine project:", gee.init_ee(EE_PROJECT or None))
    print("Earth Engine requests (cached layers are not requested again):", gee.request_plan(specs, grid))
for s in specs:
    print(f"  {s.id:<22} {s.dataset} ({s.source}, native {s.native_m:,.0f} m)")

## 4. Inputs

Each variable on the grid, with its range and the share of cells with no data, and the land cover that gives G. Look at the units: a missing or doubled scale factor shows up here.

In [ ]:
from nbs_runtime.audit import audit_data, describe
from nbs_runtime.workflow import load_inputs

inputs = load_inputs(fam, ctx, cfg.eligibility.mode, cfg.eligibility.classes, cfg.eligibility.min_share)
if boundary is not None:                       # cells outside the official boundary are not part of the area
    inside = grid.mask(boundary)
    for v in inputs.layers:
        inputs.layers[v] = np.where(inside, inputs.layers[v], np.nan)
    inputs.eligibility.G = np.where(inside, inputs.eligibility.G, np.nan)
if inputs.errors:
    print("NOT LOADED:", inputs.errors)
for v, d in inputs.data.items():
    a = audit_data(d)
    print(describe(a))
    viz.plot_variable_audit(a, grid, fig_path(f"input_{v}"), boundary)
    plt.show()

In [ ]:
e = inputs.eligibility
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
viz._map(axes[0], e.share, grid, f"Share of each cell in classes {list(e.classes)}", cmap="Greens", vmin=0, vmax=1,
         boundary=boundary)
viz._map(axes[1], e.G, grid, "G: eligible (1) or not (0)", cmap="Greys", vmin=0, vmax=1, boundary=boundary)
fig.suptitle(("SYNTHETIC TEST: " if synthetic else "") + "Eligibility", fontweight="bold")
plt.show()
for n in e.notes:
    print("note:", n)
g = e.G[~np.isnan(e.G)]
print(f"eligible: {g.mean():.1%} of the cells with land-cover data")

## 5. From value to score: z = f(x)

The grey bands are outside the evidence: no paper behind the recipe covers those values. The histogram is the distribution of the area's values, so the share of cells left unevaluated by each recipe can be seen before computing anything.

In [ ]:
from nbs_runtime.suitability import compute

viz.plot_curves(fam, inputs.layers, fig_path("curves_with_data"))
plt.show()

base = compute(fam, inputs.layers, inputs.eligibility.G, weights=cfg.run.weights, outside=cfg.run.outside)
print("share of eligible cells with data that fall outside each recipe's evidence:")
for v, s in base.outside_share(inputs.layers).items():
    r = fam.recipe(v)
    print(f"  {s:6.1%}  {v:<24} nodes {r.x[0]:g} to {r.x[-1]:g} {r.unit}")
viz.plot_scores(base, grid, synthetic, fig_path("scores"), boundary)
plt.show()

## 6. L, G, M and S

In [ ]:
n = base.S.size
print("Pixel states")
for name, c in base.status_counts().items():
    print(f"  {c / n:6.1%}  {c:>10,}  {name}")
print("\nWhich limiting variable sets L (scored cells)")
for v, c in base.driver_counts().items():
    print(f"  {c:>10,}  {v}")
print("\nModulating weights used:", {k: round(x, 3) for k, x in base.weights.items()})
for note in base.notes:
    print("note:", note)
viz.plot_index(base, grid, synthetic, fig_path("index"), boundary)
plt.show()

## 7. One pixel, step by step

The same calculation as the 12-pixel worked example, for one real cell: its values, their scores, L, M and S.

In [ ]:
from nbs_runtime.workflow import explain_pixel

if PIXEL is None:
    if base.scored.any():
        row, col = np.unravel_index(np.nanargmax(base.S), base.S.shape)
    else:
        row, col = 0, 0
else:
    row, col = PIXEL
print(explain_pixel(base, fam, inputs.layers, int(row), int(col)))

## 8. Classes

Score 0 is a class of its own; positive scores are split into quintiles over the scored area. The classes are relative to this area, so the absolute range is always shown next to them.

In [ ]:
from nbs_runtime.suitability import classify

classes = classify(base.S)
print(f"{classes.k} classes above 0" + (f" ({classes.note})" if classes.note else ""))
print("absolute S over positive scores:", {k: round(v, 3) for k, v in classes.summary.items()})
print("class limits:", np.round(classes.edges, 3))
viz.plot_classes(classes, grid, synthetic, fig_path("classes"), boundary)
plt.show()

## 9. Sensitivity

How much the map depends on the weights and on each variable. None of these tests needs anything beyond the recipes.

1. **Equal weights**: the same map with equal modulating weights (the methodology's benchmark).
2. **Dirichlet**: many draws of the modulating weights around the recipe weights. A cell in the top quintile in at least 80% of the draws is a robust hotspot.
3. **Leave one out**: the map without each variable in turn.

In [ ]:
from nbs_runtime.sensitivity import dirichlet, equal_weights, leave_one_out

eq = equal_weights(fam, inputs.layers, inputs.eligibility.G, base=base, outside=cfg.run.outside)
dres = dirichlet(fam, inputs.layers, inputs.eligibility.G, n_draws=cfg.sensitivity.n_draws,
                 concentration=cfg.sensitivity.concentration, seed=cfg.sensitivity.seed,
                 outside=cfg.run.outside, base=base)
loo = leave_one_out(fam, inputs.layers, inputs.eligibility.G, base=base, outside=cfg.run.outside)
ds = dres.summary()

print(f"1. Equal weights {eq['weights']}")
print(f"   rank agreement {eq['spearman']:.3f} | top quintile retained {eq['top_retained']:.1%} | "
      f"mean |change| {eq['mean_abs_change']:.3f}")
print(f"2. Dirichlet ({len(dres.spearman)} draws, concentration {dres.concentration:g})")
print(f"   rank agreement median {ds['spearman_median']:.3f} (5th percentile {ds['spearman_p05']:.3f}) | "
      f"robust share of the top quintile {ds['base_top_that_is_robust']:.1%} | mean 5-95% width {ds['mean_range_5_95']:.3f}")
print("3. Leave one out")
for v, o in loo.items():
    print(f"   without {v:<24} ({o['role']:<10}) rank agreement {o['spearman']:.3f} | "
          f"top retained {o['top_retained']:.1%} | newly scored cells {o['newly_scored']:,}")
viz.plot_sensitivity(base, dres, loo, grid, synthetic, fig_path("sensitivity"), boundary)
plt.show()

## 10. Outputs

`arrays.npz` (S, L, M, G, states, classes, each score, the sensitivity layers), GeoTIFFs of the suitability (bands S, L, M), the classes and the states, and `manifest.json` with everything needed to reproduce the run: the configuration, the recipe hashes, the roles and weights used, and the summaries.

In [ ]:
from nbs_runtime.workflow import save_run

manifest = {
    "synthetic": synthetic,
    "config": cfg.to_dict(),
    "plan": p.manifest(),
    "family": fam.manifest(),
    "weights_used": base.weights,
    "weight_mode": base.weight_mode,
    "outside": base.outside,
    "eligibility": {"notes": inputs.eligibility.notes},
    "inputs": {v: {"origin": d.origin, "dataset": d.spec.dataset, "error": d.error} for v, d in inputs.data.items()},
    "status_counts": base.status_counts(),
    "outside_share": base.outside_share(inputs.layers),
    "driver_counts": base.driver_counts(),
    "classes": {"k": classes.k, "edges": classes.edges, "summary": classes.summary, "note": classes.note},
    "sensitivity": {
        "equal_weights": {k: eq[k] for k in ("weights", "spearman", "top_retained", "mean_abs_change")},
        "dirichlet": {**ds, "n_draws": len(dres.spearman), "concentration": dres.concentration},
        "leave_one_out": {v: {k: o[k] for k in ("role", "spearman", "top_retained", "mean_abs_change", "newly_scored")}
                          for v, o in loo.items()},
    },
}
if SAVE:
    files = save_run(OUT_DIR, grid, manifest, base, classes,
                     extra={"sens_median": dres.median, "sens_p05": dres.p05, "sens_p95": dres.p95,
                            "sens_p_top": dres.p_top, "eligible_share": inputs.eligibility.share})
    print("written to", OUT_DIR.relative_to(REPO_ROOT))
    for f in files:
        print("  ", f.name)
else:
    print("SAVE is False: nothing written")